In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
from torchvision import models, transforms
from PIL import Image

# Define maximum images per folder
MAX_IMAGES_PER_FOLDER = 10  # Limit images processed per folder

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define model path and dataset path
checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/Improved_20250209/final_model_vgg_improved_20250209.pth'
test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ3/train/'

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define the modified VGG16 model
class VGG16Improved(nn.Module):
    def __init__(self, num_classes=2, fine_tune=True):
        super(VGG16Improved, self).__init__()
        self.vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        if fine_tune:
            for idx, child in enumerate(self.vgg.features.children()):
                if idx < 10:
                    for param in child.parameters():
                        param.requires_grad = False

        in_features = self.vgg.classifier[0].in_features
        self.vgg.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.vgg(x)

# Function to load the trained model
def load_model(checkpoint_path):
    model = VGG16Improved()
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model

# Load model
model = load_model(checkpoint_path)
print("Model loaded successfully.")

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# Function to preprocess an image
def preprocess_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    return image_tensor

# Grad-CAM function
def generate_gradcam(model, image_tensor, target_layer):
    features = []
    grads = []

    # Hook to capture features
    def forward_hook(module, input, output):
        features.append(output)

    # Hook to capture gradients
    def backward_hook(module, grad_in, grad_out):
        grads.append(grad_out[0])

    # Register hooks
    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_backward_hook(backward_hook)

    # Forward pass
    output = model(image_tensor)
    predicted_class_idx = torch.argmax(output, dim=1).item()

    # Backward pass
    model.zero_grad()
    output[0, predicted_class_idx].backward()

    # Remove hooks
    forward_handle.remove()
    backward_handle.remove()

    # Extract feature maps and gradients
    feature_map = features[0].cpu().detach().numpy()[0]
    gradient = grads[0].cpu().detach().numpy()[0]

    # Compute Grad-CAM
    weights = np.mean(gradient, axis=(1, 2))
    cam = np.zeros(feature_map.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * feature_map[i]

    cam = np.maximum(cam, 0)  # Apply ReLU
    cam = cv2.resize(cam, (224, 224))  # Resize to image size
    cam -= cam.min()
    cam /= cam.max() + 1e-8  # Normalize

    return cam, predicted_class_idx

# Function to visualize results
def visualize_results(image_path, gradcam, predicted_class):
    original_image = Image.open(image_path).resize((224, 224), Image.LANCZOS)
    original_image = np.array(original_image)

    # Overlay Grad-CAM on the image
    heatmap = cv2.applyColorMap(np.uint8(255 * gradcam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = np.clip(0.4 * heatmap + 0.6 * original_image, 0, 255).astype(np.uint8)

    # Plot images
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.title(f"Original Image\nPredicted: {class_names[predicted_class]}")
    plt.imshow(original_image)
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.title("Grad-CAM Overlay")
    plt.imshow(overlay)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Function to process test folder
def process_test_folder(test_folder_path, model):
    target_layer = model.vgg.features[-1]  # Last convolutional layer

    for subfolder in os.listdir(test_folder_path):
        subfolder_path = os.path.join(test_folder_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        print(f"\nProcessing folder: {subfolder_path}")
        image_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        image_files = image_files[:MAX_IMAGES_PER_FOLDER]

        for file in image_files:
            image_path = os.path.join(subfolder_path, file)
            print(f"\nProcessing: {image_path}")
            try:
                image_tensor = preprocess_image(image_path)

                # Generate Grad-CAM
                gradcam, predicted_class_idx = generate_gradcam(model, image_tensor, target_layer)

                # Visualize
                visualize_results(image_path, gradcam, predicted_class_idx)
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# Execute the pipeline
process_test_folder(test_folder_path, model)


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-1-65788328cd04>", line 5, in <cell line: 0>
    import torchvision.transforms as transforms
  File "/usr/local/lib/python3.11/dist-packages/torchvision/__init__.py", line 5, in <module>
    import torch
  File "/usr/local/lib/python3.11/dist-packages/torch/__init__.py", line 367, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 216, in _lock_unlock_module
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object

TypeError: object of type 'NoneType' has no len()

In [ ]:
!pip install captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from captum.attr import DeepLift
import cv2

# Define the maximum number of images to process per folder
MAX_IMAGES_PER_FOLDER = 10  # Change this value as needed

# Check for GPU availability and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Function to preprocess images
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0)
    return image_tensor.to(device)

# Define the improved VGG16 model with fine-tuning
class VGG16Improved(nn.Module):
    def __init__(self, num_classes=2, fine_tune=True):
        super(VGG16Improved, self).__init__()
        self.vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

        if fine_tune:
            for idx, child in enumerate(self.vgg.features.children()):
                if idx < 10:
                    for param in child.parameters():
                        param.requires_grad = False

        in_features = self.vgg.classifier[0].in_features
        self.vgg.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.vgg(x)

# Function to load the trained model
def load_model(checkpoint_path):
    model = VGG16Improved()
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()
    return model

# Define class names
class_names = ['Glaucous_Winged_Gull', 'Slaty_Backed_Gull']

# Function to apply DeepLIFT
def deep_lift(model, image_tensor, target_class):
    deep_lift = DeepLift(model)
    baseline = torch.zeros_like(image_tensor).to(device)
    attr, _ = deep_lift.attribute(image_tensor, baselines=baseline, target=target_class, return_convergence_delta=True)
    return attr.squeeze().cpu().detach().numpy()

# Function to process and normalize the DeepLIFT map
def process_deep_lift_map(deep_lift_map):
    deep_lift_map = np.sum(np.abs(deep_lift_map), axis=0)
    deep_lift_map = (deep_lift_map - deep_lift_map.min()) / (deep_lift_map.max() - deep_lift_map.min() + 1e-8)
    return deep_lift_map

# Function to visualize results
def plot_comparative_visualizations(image_path, deep_lift_map, predicted_class):
    image = Image.open(image_path).convert('RGB').resize((224, 224))
    image = np.array(image)
    processed_deep_lift_map = process_deep_lift_map(deep_lift_map)
    heatmap = cv2.applyColorMap(np.uint8(255 * processed_deep_lift_map), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.6 * image + 0.4 * heatmap).astype(np.uint8)

    plt.figure(figsize=(18, 6))
    plt.subplot(1, 3, 1)
    plt.imshow(image)
    plt.title(f"Original Image\nPredicted Class: {class_names[predicted_class]}")
    plt.axis('off')
    plt.subplot(1, 3, 2)
    plt.imshow(processed_deep_lift_map, cmap='hot')
    plt.title("DeepLIFT Relevance Map")
    plt.axis('off')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title("DeepLIFT Overlay")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Function to process images in all subfolders
def process_test_folder(test_folder_path, model):
    for subfolder in os.listdir(test_folder_path):
        subfolder_path = os.path.join(test_folder_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue

        print(f"Processing folder: {subfolder_path}")
        image_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))]
        image_files = image_files[:MAX_IMAGES_PER_FOLDER]

        for file in image_files:
            image_path = os.path.join(subfolder_path, file)
            print(f"\nProcessing: {image_path}")
            try:
                image_tensor = preprocess_image(image_path)
                with torch.no_grad():
                    output = model(image_tensor)
                    probabilities = torch.softmax(output, dim=1)
                    predicted_class_idx = torch.argmax(probabilities, dim=1).item()
                    predicted_class_prob = probabilities[0, predicted_class_idx].item()
                    print(f"Predicted Class: {class_names[predicted_class_idx]} (Probability: {predicted_class_prob:.4f})")
                deep_lift_map = deep_lift(model, image_tensor, predicted_class_idx)
                plot_comparative_visualizations(image_path, deep_lift_map, predicted_class_idx)
            except Exception as e:
                print(f"Error processing {image_path}: {e}")

# Main execution block
if __name__ == "__main__":
    checkpoint_path = '/content/drive/My Drive/FYP/VGGModel/Improved_20250209/final_model_vgg_improved_20250209.pth'
    model = load_model(checkpoint_path)
    print("Model loaded successfully.")
    test_folder_path = '/content/drive/My Drive/FYP/Dataset/HQ3/train/'
    process_test_folder(test_folder_path, model)


Output hidden; open in https://colab.research.google.com to view.